# [실습 12-1] Hugging Face로 맛보는 NLP 파이프라인

| 항목 | 내용 |
|---|---|
| 예상 소요 시간 | 30~40분 (CPU 가능, **GPU 권장**) |
| 본문 연계 | 12.1 토큰·임베딩 → 12.2 언어모델 |
| 선수 실습 | 없음 |
| 준비 | 부록 B.1·B.3 참고. 모델 최초 다운로드 수백 MB(1회 캐시) |
| 비용 | **완전 무료** — 유료 API 키 불요(확정 사항 ④) |

토크나이저로 문장을 조각내고(12.1), 사전학습 모델로 감성을
분류하고, 한국어 모델로 문장을 이어 쓴다(12.2) —
본문 순서 그대로 LLM의 부품을 하나씩 만져 본다.

### [준비] 환경 설정 (저장소 전용)

In [1]:
# 저장소 루트를 임포트 경로에 추가
# (Colab에서는 아래 세 줄의 주석을 해제하고 실행)
# !git clone https://github.com/tbgoodlife/ai-labs.git
# %cd ai-labs
# !pip -q install transformers==4.44.2
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "utils").exists():
    ROOT = ROOT.parent          # ch12/에서 연 경우
sys.path.insert(0, str(ROOT))

import os
os.environ["USE_TF"] = "0"   # PyTorch 경로 고정

import platform
import transformers
from transformers import set_seed
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
from utils import plot_style

plot_style.apply()              # 도해 스타일 킷 적용
print("Python", platform.python_version())
print("transformers", transformers.__version__)
set_seed(42)                    # 생성(셀 3) 재현 조건

/Users/jungsookim/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python 3.12.6
transformers 4.44.2


### [셀 1] 토크나이저 — 문장은 어떻게 조각나는가 📖

In [2]:
from transformers import AutoTokenizer

en = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased")
ko = AutoTokenizer.from_pretrained(
    "skt/kogpt2-base-v2")

text_en = "Artificial intelligence is everywhere."
text_ko = "인공지능은 이미 어디에나 있다."
print("영어:", en.tokenize(text_en))
print("한국어:", ko.tokenize(text_ko))
print(f"토큰 수: 영어 {len(en.tokenize(text_en))} / "
      f"한국어 {len(ko.tokenize(text_ko))}")

/Users/jungsookim/Documents/Claude/Book/AI/notebooks/.venv-hf44/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


영어: ['artificial', 'intelligence', 'is', 'everywhere', '.']
한국어: ['▁인공', '지', '능', '은', '▁이미', '▁어디에', '나', '▁있다.']
토큰 수: 영어 5 / 한국어 8


**핵심 포인트**
- 같은 뜻의 문장인데 한국어가 훨씬 **잘게, 많이** 쪼개진다 — 영어 중심 토크나이저에게 한국어는 낯선 언어라서다(심화 박스 "한국어와 LLM"의 실물 근거).
- 토큰 수 = 비용·문맥 창 소모량 — 12.1 〈토큰화〉에서 배운 "토큰이 화폐 단위"가 여기서 눈에 보인다.

실패 시 대처: 최초 다운로드 지연은 정상(1회 캐시 후 즉시).

### [셀 2] 감성 분석 — 사전학습 모델 바로 쓰기 📖

In [3]:
from transformers import pipeline

clf = pipeline("sentiment-analysis")   # 기본 모델 로드
reviews = [
    "This movie was absolutely wonderful!",
    "Terrible service. Never coming back.",
    "The plot was fine, nothing special.",
]
for r in reviews:
    out = clf(r)[0]
    print(f"{out['label']:8s} {out['score']:.3f}  {r}")

POSITIVE 1.000  This movie was absolutely wonderful!
NEGATIVE 0.996  Terrible service. Never coming back.
NEGATIVE 0.968  The plot was fine, nothing special.


/Users/jungsookim/Documents/Claude/Book/AI/notebooks/.venv-hf44/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


**핵심 포인트**
- 코드 세 줄로 감성 분류기가 생겼다 — 사전학습(12.2)의 힘이자 전이학습(10장)의 NLP판이다.
- 세 번째 문장("not bad")까지 긍정으로 맞히는지 확인하자 — 단어가 아니라 **문맥**을 읽는다는 증거.
- **확신도(score)가 높다고 항상 옳은 것은 아니다** — 모델의 확신과 사실은 별개다(12.4 할루시네이션의 복선).

### [셀 3] 텍스트 생성 — 다음 토큰 예측의 실물 📖

In [4]:
gen = pipeline("text-generation",
               model="skt/kogpt2-base-v2")

prompt = "인공지능 기술의 발전으로 우리의 일상은"
out = gen(prompt, max_length=50, do_sample=True,
          temperature=0.8, top_k=50,
          pad_token_id=gen.tokenizer.eos_token_id)
print(out[0]["generated_text"])

인공지능 기술의 발전으로 우리의 일상은 점점 더 복잡해지고 있다"고 설명했다.
이번 세미나에는 인공지능의 발달과정과 활용방법에 관한 논의가 진행됐다. 이번 '2017 PSN 월드 챔피언십'은 오는 11월 14일부터 15일까지


**핵심 포인트**
- 생성 = "다음 토큰 예측"의 반복(12.2)이다 — 이 작은 한국어 모델(1.25억 파라미터)도 원리는 최신 LLM과 같다.
- 출력이 **그럴듯하지만 사실 보장은 없다** — 문장이 자연스러울수록 위험한 이유(12.4 할루시네이션의 원인)를 눈으로 확인하자.
- `temperature`·`top_k`가 창의성 손잡이다 — [심화 1]에서 조절해 본다.

실패 시 대처: 시드를 고정해도 라이브러리 버전에 따라 출력이 다를 수 있다 — 문장이 달라도 "그럴듯한 이어 쓰기"라는 관찰은 동일하다.

### [보조 1] 한국어 감성 분석 — 한국어로 배운 모델

In [5]:
clf_ko = pipeline("sentiment-analysis", model=(
    "WhitePeak/bert-base-cased-Korean-sentiment"))

for r in ["배송이 빠르고 품질도 좋아요",
          "두 번 다시 안 삽니다"]:
    out = clf_ko(r)[0]
    print(f"{out['label']:<9}"
          f"(확신도 {out['score']:.3f})  {r}")
# 한국어 데이터로 학습된 모델은 한국어를 제대로 읽는다 —
# [셀 1]의 토큰 문제와 함께 "언어 자원 격차"를 생각해 보자

LABEL_1  (확신도 0.971)  배송이 빠르고 품질도 좋아요
LABEL_0  (확신도 0.997)  두 번 다시 안 삽니다


### [보조 2] 임베딩 유사도 미니 데모 (12.1 · 실습 12-3 준비)

In [6]:
import numpy as np
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained(
    "distilbert-base-uncased")
tok = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased")

def embed(text):
    """문장 → 벡터(마지막 층 평균)."""
    with torch.no_grad():
        out = model(**tok(text, return_tensors="pt"))
    return out.last_hidden_state.mean(1)[0].numpy()

def cos(a, b):
    return a @ b / (np.linalg.norm(a)
                    * np.linalg.norm(b))

base = embed("A cat sits on the mat.")
for s in ["A kitten rests on the rug.",
          "The stock market fell sharply."]:
    print(f"{cos(base, embed(s)):.3f}  {s}")
# 뜻이 가까우면 벡터도 가깝다 — 12.4 RAG 검색의
# 원리이며, 실습 12-3에서 그대로 재사용한다

0.952  A kitten rests on the rug.
0.643  The stock market fell sharply.


/Users/jungsookim/Documents/Claude/Book/AI/notebooks/.venv-hf44/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### [심화 1] 생성 조절 실험 (12.2 연동)

In [7]:
# TODO: temperature 0.2 / 0.8 / 1.5로 [셀 3]을
#       재실행해 "안전한 반복 ↔ 창의적 헛소리"
#       스펙트럼을 기록하자. max_length·top_k도
#       바꿔 보고, 안전한 시작값을 스스로 정해 보자.
for t in [0.2, 1.5]:
    set_seed(42)
    out = gen(prompt, max_length=50, do_sample=True,
              temperature=t, top_k=50,
              pad_token_id=gen.tokenizer.eos_token_id)
    print(f"[T={t}]", out[0]["generated_text"], "\n")

[T=0.2] 인공지능 기술의 발전으로 우리의 일상은 점점 더 복잡해지고 있다.
이러한 상황에서 우리 사회도 인공지능의 활용에 대한 사회적 논의가 활발하게 진행되고 있다.
인공지능은 인간의 뇌를 모방한 인공지능의 한 분야다.
인공 



[T=1.5] 인공지능 기술의 발전으로 우리의 일상은 이제 더 다양한 방법으로 변화해 나가면 좋겠다"고 말했다. 가을의 끝자락에서 가을로 진입했다는 얘기, 다시 말하자면 이번 주말이 마지막 일요일인 셈이다. 코웨이의 대표 청소기 



---
## 마무리

- 토큰화 → 사전학습 모델 활용 → 생성까지, 12.1~12.2의 개념이 각각 코드 몇 줄이었다.
- 한국어 토큰 비효율([셀 1])과 한국어 전용 모델([보조 1])—언어 자원 격차는 기술이자 사회 문제다.
- 확신도≠정확성, 유창함≠사실 — 12.4의 한계 논의로 가는 다리를 실물로 건넜다.

**연습문제 연계**: [응용] 벡터 유추 문항의 감각은 [보조 2]에서, [심화] 생성 조절 해석은 [심화 1]에서 기른다.

**다음 실습**: [실습 12-2] 프롬프트 Before/After (`lab-12-02_prompt-before-after.ipynb`)